# Thetis — Treino SAFSAR Few-Shot no Kaggle

Meta-treino episódico do **SAFSAR** (*Semantic-Aware Few-Shot Action
Recognition*, Tang et al., WACV 2024) com backbone **VideoMAE-Base** (3D) e
descrições dos golpes codificadas pelo **BERT-Base** congelado, sobre o dataset
[THETIS](https://github.com/THETIS-dataset/dataset), no protocolo 5-way K-shot
(val/test em 3-way, conforme o split 6/3/3).

Serve **qualquer uma das 5 modalidades** × {1-shot, 5-shot} — escolha em
`MODALITY` e `K_SHOT` na seção 1, que o resto do notebook se ajusta sozinho.

## O que muda em relação aos outros notebooks

Splits, seed, episódios e protocolo de avaliação são os mesmos. Frames por clipe
(8), amostragem temporal (`segment`) e resolução (112²) são as do TRX e do
MVP-Shot. Muda o que é do método, e tudo vem de `experiments/configs/safsar_*.yaml`:

| | MVP-Shot | SAFSAR |
|---|---|---|
| Encoder | CLIP ResNet-50, um vetor por frame | VideoMAE-Base (Kinetics-400), um vetor por vídeo (3D) |
| Texto | nome da classe (CLIP), só no suporte | descrição do golpe (BERT, congelado), só no suporte |
| Cabeça | PSTI + OTAM em 3 velocidades | fusão protótipo + texto (2 camadas), TLM e cosseno |
| Perda | entropia cruzada | entropia cruzada + classificador global das 6 classes de treino |

O artigo não publicou código: as escolhas que ele deixa em aberto estão em
`experiments/configs/README.md` (seção "SAFSAR").

## Antes de rodar — checklist do painel direito (⚙️ *Session options*)

| Opção | Valor |
|---|---|
| **Accelerator** | `GPU T4 x2` ou `GPU P100` (o código usa **uma** GPU só) |
| **Internet** | `On` — obrigatório (clone do repo, `pip`, pesos do VideoMAE e do BERT, do Hugging Face). Exige conta verificada por telefone. |
| **Persistence** | `Files only` (ou *Variables and Files*) — é o que mantém `/kaggle/working` entre sessões e permite retomar o treino |
| **Input** | um Kaggle Dataset com a pasta da modalidade escolhida (o mesmo dos outros notebooks serve) |

**Dados:** crie um dataset no Kaggle (*Datasets → New Dataset*) subindo o zip da
pasta da modalidade — o Kaggle descompacta sozinho, e o resultado fica em
`/kaggle/input/<slug>/<PASTA>/<classe>/*.avi`. Gere o zip localmente com:

```bash
cd dataset && zip -r -0 VIDEO_RGB.zip VIDEO_RGB       # rgb
# ou VIDEO_Depth / VIDEO_Mask / VIDEO_Skelet2D / VIDEO_Skelet3D
```

| `MODALITY` | Pasta no dataset | Clipes `.avi` |
|---|---|---|
| `rgb` | `VIDEO_RGB` | 1980 |
| `depth` | `VIDEO_Depth` | 1980 |
| `mask` | `VIDEO_Mask` | 1980 |
| `skeleton_2d` | `VIDEO_Skelet2D` | 1216 |
| `skeleton_3d` | `VIDEO_Skelet3D` | 1217 |

O `manifest.csv` **não** precisa ser subido: a seção 4 o regenera a partir da
própria árvore (os splits de classe são determinísticos pela seed, então saem
idênticos aos do repositório).

**Código:** este notebook clona o repositório do GitHub — faça `git push` das
suas mudanças locais antes de rodar aqui.

## Sobre o limite de sessão (leia antes de escolher `EPOCHS`)

A sessão do Kaggle é cortada em ~12 h, e um commit que estoura o tempo **falha
sem salvar o output**. O ritmo do SAFSAR **ainda não foi medido**. O VideoMAE-B a
112² e 8 frames custa ~2× a ResNet-50 do TRX, que mediu ~300 s por época no
1-shot e ~500 s no 5-shot — então espere algo como ~600 s e ~1000 s (de 8 a 14 h
para as 50 épocas). Confira o tempo da 1ª época (aparece no fim de cada linha de
resumo) antes de escolher o `EPOCHS` da próxima sessão.

O trainer grava `last.pt` + `training.json` **a cada época** e aceita
`--resume`, então dá para treinar **em blocos**: `EPOCHS=20` na primeira sessão,
`EPOCHS=40` na segunda, `EPOCHS=50` na terceira — a célula de treino sempre
continua da época seguinte à última salva.

## 0. Ambiente: GPU, RAM, disco e internet

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader \
    || echo 'SEM GPU — ative em Session options > Accelerator > GPU'

import os, shutil, socket
import torch

print('torch', torch.__version__, '| CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB')

ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
free = shutil.disk_usage('/kaggle/working').free / 1e9
print(f'RAM: {ram:.1f} GB | livre em /kaggle/working: {free:.1f} GB')

try:
    socket.create_connection(('pypi.org', 443), timeout=5)
    print('Internet: ON')
except OSError:
    print('Internet: OFF  <-- ligue em Session options > Internet (sem isso o clone e os pesos pré-treinados falham)')

## 1. Configuração

`EPOCHS` é a **meta acumulada** desta sessão, não um incremento: com `--resume`,
o treino vai da última época salva até `EPOCHS`. Comece com 20.

In [ ]:
REPO_URL = 'https://github.com/CauBitten/thetis.git'
REPO_DIR = '/kaggle/working/Thetis'

# –– Experimento ––––––––––––––––––––––––––––––––––––––––––––––––––
MODALITY = 'rgb'   # rgb | depth | mask | skeleton_2d | skeleton_3d
K_SHOT   = 1       # 1 ou 5

_MOD = {
    'rgb':         ('VIDEO_RGB',      'rgb',        1980),
    'depth':       ('VIDEO_Depth',    'depth',      1980),
    'mask':        ('VIDEO_Mask',     'mask',       1980),
    'skeleton_2d': ('VIDEO_Skelet2D', 'skeleton2d', 1217),
    'skeleton_3d': ('VIDEO_Skelet3D', 'skeleton3d', 1217),
}
assert MODALITY in _MOD, f'MODALITY invalida: {MODALITY!r} -- use um de {list(_MOD)}'
assert K_SHOT in (1, 5), f'K_SHOT deve ser 1 ou 5, nao {K_SHOT}'
VIDEO_DIR, _slug, N_AVI_ESPERADO = _MOD[MODALITY]

BASE_CONFIG = f'experiments/configs/safsar_{_slug}_5w{K_SHOT}s.yaml'
RUN_ID      = f'safsar_{_slug}_5w{K_SHOT}s_kaggle'

# None = procurar sozinho em /kaggle/input (desce ate 5 niveis).
DATASET_ROOT = None

WORK_ROOT   = '/kaggle/working'
OUTPUT_ROOT = f'{WORK_ROOT}/outputs'   # checkpoints + resultados da avaliacao
LOG_ROOT    = f'{WORK_ROOT}/logs'      # training.json
DATA_ROOT   = f'{WORK_ROOT}/data'      # manifest gerado aqui

EPOCHS = 20             # 20 -> 40 -> 50
GRAD_CHECKPOINT = True  # mantenha True: e o valor de todos os configs (o VideoMAE nao tem
                        # BatchNorm, entao aqui ele so economiza memoria)

print(f'{MODALITY} / {K_SHOT}-shot -> {BASE_CONFIG}')
print(f'pasta esperada no dataset: {VIDEO_DIR}/  ({N_AVI_ESPERADO} clipes)')
print(f'run_id: {RUN_ID}')

## 2. Clonar o repositório e instalar dependências

In [ ]:
import os, subprocess
from pathlib import Path

if Path(REPO_DIR, '.git').is_dir():
    # sincroniza com o remoto sem quebrar em clone raso (não apaga arquivos não versionados)
    !git -C {REPO_DIR} fetch --depth 1 origin main && git -C {REPO_DIR} reset --hard origin/main
else:
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -1

# Este notebook depende do SAFSAR registrado e do --resume no trainer; se faltar, é push faltando.
help_txt = subprocess.run(
    ['python', 'src/training/meta_trainer.py', '--help'], capture_output=True, text=True
).stdout
methods = subprocess.run(
    ['python', '-c', 'from src.models.factory import METHODS; print(sorted(METHODS))'],
    capture_output=True, text=True,
)
assert '--resume' in help_txt and "'safsar'" in methods.stdout, (
    'O repositório clonado não tem o SAFSAR (src/models/safsar.py registrado em src/models/factory.py) '
    'ou o --resume no trainer. Faça commit + push e rode esta célula de novo.\n'
    f'{methods.stdout}{methods.stderr}'
)
print('OK — trainer com --resume | métodos:', methods.stdout.strip())

In [ ]:
%pip install -q decord || echo 'decord não instalou — segue com OpenCV'
%pip install -q 'transformers>=4.45'   # VideoMAE + BERT do SAFSAR

import importlib
faltando = []
for m in ['torch', 'torchvision', 'cv2', 'pandas', 'numpy', 'yaml', 'scipy', 'matplotlib', 'transformers']:
    try:
        importlib.import_module(m)
    except Exception as e:
        faltando.append((m, str(e)))
print('OK — dependências obrigatórias presentes' if not faltando else f'FALTANDO: {faltando}')

## 3. Localizar os dados

Procura a pasta que contém `{VIDEO_DIR}/` (definida por `MODALITY` na seção 1)
entre os datasets anexados e confere se o `decord` realmente decodifica um clipe
(se não, ele é removido e o loader usa o OpenCV — mais lento apenas na 1ª época,
já que o decode é cacheado em RAM).

In [ ]:
from pathlib import Path

def _find_modality_root(video_dir, explicit=None, base='/kaggle/input', max_depth=5):
    '''Retorna a pasta que CONTEM video_dir/, varrendo /kaggle/input em largura.

    O Kaggle monta o dataset em profundidades diferentes conforme como ele foi
    anexado (/kaggle/input/<slug>/ ou /kaggle/input/datasets/<user>/<slug>/),
    entao vale descer alguns niveis em vez de assumir um layout.
    '''
    if explicit:
        p = Path(explicit)
        if p.name == video_dir:            # apontaram para dentro da pasta
            p = p.parent
        return p if Path(p, video_dir).is_dir() else None

    base = Path(base)
    if not base.is_dir():
        return None
    nivel = [base]
    for _ in range(max_depth):
        proximo = []
        for d in nivel:
            try:
                filhos = sorted(p for p in d.iterdir() if p.is_dir())
            except PermissionError:
                continue
            if any(p.name == video_dir for p in filhos):
                return d
            proximo.extend(p for p in filhos if p.name != video_dir)
        nivel = proximo
        if not nivel:
            break
    return None

DATASET_ROOT = _find_modality_root(VIDEO_DIR, DATASET_ROOT)
if DATASET_ROOT is None:
    visto = sorted(str(p) for p in Path('/kaggle/input').glob('*')) if Path('/kaggle/input').is_dir() else []
    raise AssertionError(
        f'Nao achei nenhuma pasta {VIDEO_DIR} em /kaggle/input.\n'
        f'O que existe em /kaggle/input: {visto}\n'
        f"Anexe o dataset da modalidade '{MODALITY}' em '+ Add Input' ou preencha "
        f'DATASET_ROOT na secao 1 (pode colar o caminho ate {VIDEO_DIR} -- o notebook usa a pasta acima).'
    )

n_avi = sum(1 for _ in Path(DATASET_ROOT, VIDEO_DIR).rglob('*.avi'))
print(f'DATASET_ROOT = {DATASET_ROOT}  |  {n_avi} videos .avi em {VIDEO_DIR}/')
if n_avi < N_AVI_ESPERADO:
    print(f"AVISO: esperado {N_AVI_ESPERADO} clipes para '{MODALITY}'; achei {n_avi}. Upload incompleto?")

In [ ]:
sample = next(Path(DATASET_ROOT, VIDEO_DIR).rglob('*.avi'))
try:
    import decord
    decord.VideoReader(str(sample))
    print('decord OK -- decode rapido na 1a epoca')
except Exception as e:
    print(f'decord indisponivel ({type(e).__name__}: {e}) -- removendo para o loader cair no OpenCV')

## 4. Gerar o `manifest.csv` e a lista de exclusões

Varre a árvore e monta o manifesto. As colunas das modalidades que você não
anexou ficam vazias — o treino usa só `path_<MODALITY>`, e o sampler descarta as
linhas onde ela está vazia.

O `--full-integrity` é o que gera o `excluded_clips.json`, que remove do pool os
clipes defeituosos que o THETIS traz: vídeos totalmente em branco (a segmentação
do Kinect falha e grava máscara preta) e duplicatas byte a byte (repetições
inexistentes preenchidas com cópia de outro take, o que deixaria o mesmo clipe no
suporte **e** na consulta do mesmo episódio). As checagens rodam só sobre a
modalidade anexada, e o resultado é idêntico ao do dataset completo. Custa alguns
segundos — **sem ele o treino roda sem filtro** e avisa no log.

In [ ]:
!python src/data/loader.py --input '{DATASET_ROOT}' --output '{DATA_ROOT}' --seed 42 --full-integrity

import json
from pathlib import Path

import pandas as pd
MANIFEST_PATH = f'{DATA_ROOT}/processed/manifest.csv'
_df = pd.read_csv(MANIFEST_PATH)
_col = f'path_{MODALITY}'
_com = (_df[_col].fillna('') != '').sum()
print(f'\nmanifest: {len(_df)} linhas | {_com} com {MODALITY} '
      f"| {_df['action_label'].nunique()} classes")
assert _com >= N_AVI_ESPERADO, (
    f'so {_com} linhas tem {_col} preenchido (esperado {N_AVI_ESPERADO}). '
    f'O dataset anexado tem a pasta {VIDEO_DIR}/ completa?'
)

_exc = json.loads(Path(f'{DATA_ROOT}/processed/excluded_clips.json').read_text())
_n = _exc['counts'][MODALITY]
print(f'exclusoes para {MODALITY}: {_n} clipes ({_com - _n} sobram no pool)')
assert _n > 0 or MODALITY not in ('rgb', 'depth', 'mask', 'skeleton_2d', 'skeleton_3d'), (
    'lista de exclusao vazia -- o --full-integrity rodou?'
)

## 5. Config derivada para o Kaggle

Parte do config da modalidade escolhida e ajusta **só o que é específico da
máquina**: caminhos dos dados, número de épocas e destinos de checkpoint/log.

> `encoder.batch_size`, `encoder.gradient_checkpointing` e `optim.stream_query`
> **fazem parte do protocolo** e não são escalados pela VRAM. Nos outros métodos
> eles mudam o resultado, por causa do BatchNorm; o VideoMAE só tem LayerNorm, então
> aqui eles só mudam a memória. Mesmo assim vêm do config (16 / `true` / `true`, os
> mesmos dos outros métodos), para manter o protocolo idêntico.
>
> O config também lista em `model.train_classes` as 6 classes de `meta_train` (as
> linhas do classificador global); o trainer confere essa lista com o split e para
> logo no início se não bater. Detalhes em `experiments/configs/README.md`.

In [ ]:
import yaml, torch
from pathlib import Path

cfg = yaml.safe_load(Path(BASE_CONFIG).read_text())

cfg['data']['manifest_path'] = str(Path(MANIFEST_PATH).resolve())
cfg['data']['dataset_root']  = str(Path(DATASET_ROOT).resolve())

cfg['encoder']['gradient_checkpointing'] = bool(GRAD_CHECKPOINT)
cfg['optim']['epochs'] = int(EPOCHS)
cfg['output_root'] = OUTPUT_ROOT
cfg['log_root']    = LOG_ROOT
cfg['run_id']      = RUN_ID

KAGGLE_CONFIG = 'experiments/configs/_kaggle_active.yaml'
Path(KAGGLE_CONFIG).write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))

vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
bs = cfg['encoder']['batch_size']
assert cfg['method'] == 'safsar', f"config {BASE_CONFIG} tem method={cfg['method']!r}, esperado 'safsar'"
assert cfg['modalities'] == [MODALITY], (
    f"config {BASE_CONFIG} tem modalities={cfg['modalities']}, esperado ['{MODALITY}']"
)
print(f'VRAM={vram:.1f} GB | encoder.batch_size={bs} (do config, NAO auto-escalado) | epochs={EPOCHS}')
if vram and vram < 12:
    print(f'  AVISO: batch_size={bs} pode dar OOM em {vram:.1f} GB. Prefira uma GPU maior a')
    print('  baixar o valor -- baixar muda o regime de BatchNorm e invalida a comparacao.')
print()
print(Path(KAGGLE_CONFIG).read_text())

## 6. Retomar de uma sessão anterior (opcional)

Com **Persistence: Files** ligada, `/kaggle/working` já volta pronto e não há
nada a fazer aqui. Se você em vez disso salvou uma versão e anexou o *output*
daquele notebook como input, esta célula copia de lá o `last.pt` e o `best.pt`.

O `best.pt` é copiado junto de propósito: ao retomar, o trainer restaura o
`best_val` do log e só regrava `best.pt` quando uma época futura supera esse
valor. Numa retomada já perto da convergência isso pode nunca acontecer, e sem
a cópia a seção 9 não teria checkpoint para avaliar.

In [ ]:
import shutil, torch
from pathlib import Path

CKPT_DIR = Path(OUTPUT_ROOT, 'checkpoints', RUN_ID)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

for nome in ('last.pt', 'best.pt'):
    destino = CKPT_DIR / nome
    if destino.exists() or not Path('/kaggle/input').is_dir():
        continue
    achados = sorted(Path('/kaggle/input').glob(f'**/checkpoints/{RUN_ID}/{nome}'))
    if achados:
        shutil.copy2(achados[0], destino)
        print(f'copiado de {achados[0]}')

last = CKPT_DIR / 'last.pt'
if last.exists():
    st = torch.load(last, map_location='cpu', weights_only=False)
    lg = st.get('log', {})
    print(f"checkpoint encontrado: época {st['epoch']}/{EPOCHS} "
          f"| best_val={lg.get('best_val_acc')} @ep{lg.get('best_epoch')}")
else:
    print('nenhum checkpoint anterior — o treino começa da época 1')

if (CKPT_DIR / 'best.pt').exists():
    print('best.pt presente — a seção 9 pode rodar a qualquer momento')
else:
    print('best.pt ausente — a seção 9 só roda depois da 1ª validação desta sessão')

## 7. Teste de sanidade (smoke)

Roda o pipeline ponta-a-ponta em segundos (encoder aleatório, 1 época, dims
mínimas). Pega erro de caminho/dados **antes** do treino longo. Não mexe no
checkpoint do treino de verdade (o `run_id` vira `smoke_...`).

In [ ]:
!python src/training/meta_trainer.py --config {KAGGLE_CONFIG} --smoke

## 8. Treino

`--resume` continua da época seguinte à do `last.pt` (e começa do zero se não
houver nenhum). A cada época o trainer regrava `last.pt` e `training.json`, e
salva `best.pt` sempre que a acurácia de validação melhora — então uma sessão
interrompida custa no máximo uma época.

A 1ª época é a mais lenta: cada clipe é decodificado uma vez e vai para o cache
em RAM (32 frames por clipe, o mesmo custo de cache do ProtoNet). Na 1ª execução
também são baixados do Hugging Face os pesos do VideoMAE (~350 MB) e do BERT
(~440 MB), e o BERT codifica as descrições das 12 classes — alguns segundos antes
da 1ª época. O tempo de cada época aparece no fim da linha de resumo (`... | 250s`) —
use isso para dimensionar o `EPOCHS` da próxima sessão.

> Para o treino longo, prefira **Save Version → Save & Run All (Commit)**: a
> sessão interativa cai se você fechar a aba, o commit roda em background.
> Só não deixe a meta passar do que cabe em ~12 h — um commit que estoura o
> tempo falha **sem** salvar o output.

Dica: `THETIS_MEM_PROFILE=1` antes do comando imprime o pico de VRAM dos 3
primeiros episódios. Se sobrar bastante folga, desligar `GRAD_CHECKPOINT` deixa o
treino ~25-30% mais rápido sem mudar o resultado (o VideoMAE não tem BatchNorm).

In [ ]:
!python src/training/meta_trainer.py --config {KAGGLE_CONFIG} --resume

## 9. Avaliação no meta_test (1000 episódios)

Roda sobre o `best.pt`. Se o treino ainda não chegou na meta de épocas, isso
continua válido — é só a avaliação do melhor checkpoint até agora.

São 3-way × (K support + 15 query) × 1000 episódios: ~48 mil clipes no 1-shot,
~60 mil no 5-shot, cada um com 8 frames pelo VideoMAE — dezenas de minutos numa
T4. Depois roda os dois subgrupos de robustez à velocidade: suporte de iniciantes
(p1–p31) e consulta de especialistas (p32–p55), e o contrário. Isso triplica o
tempo da seção. Para reavaliar uma run já treinada, rode o notebook com o mesmo
`EPOCHS`: o treino vê que já chegou na meta e não faz nada. Para uma prévia rápida,
acrescente `--n-episodes 200` ao comando.

In [ ]:
from pathlib import Path
ckpt = f'{OUTPUT_ROOT}/checkpoints/{RUN_ID}/best.pt'
assert Path(ckpt).exists(), f'checkpoint não encontrado: {ckpt} (o treino chegou a validar?)'
!python src/training/eval_episodic.py --checkpoint '{ckpt}' --output-root '{OUTPUT_ROOT}'

# Robustez à velocidade (pergunta de pesquisa 2): suporte de um nível técnico, consulta do outro.
# Grava metrics_<modo>.json e confusion_<modo>.png ao lado dos arquivos acima.
for modo in ('beginner_to_expert', 'expert_to_beginner'):
    !python src/training/eval_episodic.py --checkpoint '{ckpt}' --output-root '{OUTPUT_ROOT}' --speed-split {modo}

## 10. Curvas de treino (acurácia + loss) e matriz de confusão

In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

log = json.loads(Path(f'{LOG_ROOT}/{RUN_ID}/training.json').read_text())
E = log['epochs']
ep        = [e['epoch'] for e in E]
train_acc = [e['train_acc'] for e in E]
train_los = [e['train_loss'] for e in E]
# val_acc e val_loss só existem nas épocas de eval; mantidos alinhados por época
val = [(e['epoch'], e.get('val_acc'), e.get('val_loss')) for e in E if 'val_acc' in e]

fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(12, 4))

ax_acc.plot(ep, train_acc, label='train_acc', alpha=.85)
if val:
    ax_acc.plot([x for x, _, _ in val], [a for _, a, _ in val], 'o-', label='val_acc')
best = log.get('best_epoch')
if best is not None:
    ax_acc.axvline(best, ls='--', c='gray', alpha=.6, label=f'best @ep{best}')
ax_acc.set_xlabel('época'); ax_acc.set_ylabel('acurácia'); ax_acc.grid(alpha=.3); ax_acc.legend()

ax_loss.plot(ep, train_los, label='train_loss', alpha=.85)
vl = [(x, l) for x, _, l in val if l is not None]
if vl:
    ax_loss.plot([x for x, _ in vl], [l for _, l in vl], 'o-', label='val_loss')
ax_loss.set_xlabel('época'); ax_loss.set_ylabel('loss'); ax_loss.grid(alpha=.3); ax_loss.legend()

fig.suptitle(f"{RUN_ID} — best_val_acc={log.get('best_val_acc')} @ época {log.get('best_epoch')}")
fig.tight_layout(); plt.show()

secs = [e['seconds'] for e in E if 'seconds' in e]
if secs:
    print(f'{len(ep)} épocas registradas | média {sum(secs)/len(secs):.0f}s/época '
          f'| ~{sum(secs)/3600:.1f}h de GPU acumuladas')

In [ ]:
from IPython.display import Image, display
from pathlib import Path
png = f'{OUTPUT_ROOT}/results/{RUN_ID}/confusion.png'
display(Image(filename=png)) if Path(png).exists() else print('confusion.png ainda não gerado')

## 11. Continuar numa próxima sessão

**Com `Persistence: Files only`** (mais simples): reabra o notebook, aumente
`EPOCHS` na seção 1 e rode tudo — `/kaggle/working` voltou com os checkpoints e
a seção 8 retoma sozinha.

**Sem persistence** (ou entre notebooks diferentes):

1. *Save Version → Save & Run All (Commit)* nesta sessão;
2. na próxima, **+ Add Input → Your Work →** o output desta versão;
3. aumente `EPOCHS` e rode tudo: a seção 6 acha o `last.pt` e o `best.pt` no input e copia.

**Baixar os pesos:** painel *Output* → `outputs/checkpoints/<run_id>/best.pt`
(~1,3 GB: modelo + estado do Adam) e `logs/<run_id>/training.json`.

Para trazer os resultados de volta ao repositório, commite o `training.json` e o
`outputs/results/<run_id>/metrics.json` — são leves e versionados.